# LeetCode 564 — Find the Closest Palindrome bằng Python

- Link: https://leetcode.com/problems/find-the-closest-palindrome/
- Độ khó: **Hard**
- Chủ đề: Toán, Xử lý chuỗi, Liệt kê ứng viên (candidate enumeration), Trường hợp biên

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def nearestPalindromic(self, n: str) -> str:

```

## 1. Giới thiệu & mục tiêu học

Bài này nhìn thì đơn giản ("tìm số đối xứng gần nhất") nhưng là bài Hard vì **các trường hợp biên** rất dễ trượt: `"1"`, `"10"`, `"11"`, `"99"`, `"100"`, `"999"`, `"1001"`... Ý tưởng chính là: **đừng tìm, hãy liệt kê một vài ứng viên chắc chắn chứa đáp án**.

Sau notebook này bạn sẽ:

1. Hiểu cấu trúc của số palindrome: **nửa đầu quyết định cả số** ("soi gương").
2. Biết vì sao "soi gương nửa đầu" thôi là **chưa đủ**, và cần thêm những ứng viên nào.
3. Viết được 3 lời giải: tìm lan ra hai phía (brute force), **5 ứng viên**, và **palindrome liền trước / liền sau**.
4. Xử lý chuẩn các trường hợp "số đổi độ dài" (`99 → 101`, `100 → 99`).
5. Kiểm chứng toàn diện bằng brute force: duyệt hết mọi `n` nhỏ và nhiều `n` ngẫu nhiên.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho chuỗi `n` biểu diễn một số nguyên dương. Tìm số nguyên **palindrome** (đọc xuôi hay ngược như nhau) **gần `n` nhất**, **không tính chính `n`**. "Gần nhất" nghĩa là hiệu tuyệt đối nhỏ nhất. Nếu có hai số cách đều, trả về **số nhỏ hơn**. Kết quả trả về dạng chuỗi.

**Ví dụ 1**

```text
Input:  n = "123"
Output: "121"
```

**Ví dụ 2**

```text
Input:  n = "1"
Output: "0"
Giải thích: 0 và 2 đều cách 1 một đơn vị; chọn số nhỏ hơn là 0.
```

**Ràng buộc**

- `1 <= n.length <= 18`
- `n` chỉ gồm chữ số, không có số 0 ở đầu.
- `n` biểu diễn số nguyên trong khoảng `[1, 10^18 - 1]`.

**Vì sao không duyệt từng số được?** Khoảng cách giữa hai palindrome liên tiếp có thể rất lớn: giữa `10^17 + ...` các palindrome 18 chữ số cách nhau cỡ `10^8`–`10^9`. Duyệt từng số là quá chậm.

In [1]:
def is_pal(x: int) -> bool:
    s = str(x)
    return s == s[::-1]

for x in [0, 7, 121, 123, 1001, 1010]:
    print(x, is_pal(x))

# Ví dụ 1 bằng tay: tìm quanh 123
print([x for x in range(110, 135) if is_pal(x)])

0 True
7 True
121 True
123 False
1001 True
1010 False
[111, 121, 131]


## 3. Kiến thức nền

### 3.1. Đảo chuỗi bằng slicing `s[::-1]`

`s[::-1]` đọc chuỗi từ cuối về đầu. `s[:-1]` bỏ ký tự cuối. Kết hợp: `s[:-1][::-1]` = đảo chuỗi sau khi bỏ ký tự cuối — dùng để soi gương số có **độ dài lẻ**.

In [2]:
s = "123"
print(s[::-1], "|", s[:-1], "|", s[:-1][::-1])
print("gương chẵn :", s + s[::-1])        # 123321
print("gương lẻ   :", s + s[:-1][::-1])   # 12321

321 | 12 | 21
gương chẵn : 123321
gương lẻ   : 12321


### 3.2. Chuyển đổi `int` ↔ `str` và chia lấy phần nguyên

- `int("0123")` = 123 (bỏ số 0 đầu); `str(123)` = `"123"`.
- `(L + 1) // 2` = số chữ số của **nửa đầu** (kể cả chữ số giữa nếu `L` lẻ): `L = 5 → 3`, `L = 4 → 2`.

In [3]:
for L in range(1, 7):
    print(f"L={L}: nửa đầu dài {(L + 1) // 2}, phần gương dài {L // 2}")
print(int("0099"), str(99), len(str(10**17)))

L=1: nửa đầu dài 1, phần gương dài 0
L=2: nửa đầu dài 1, phần gương dài 1
L=3: nửa đầu dài 2, phần gương dài 1
L=4: nửa đầu dài 2, phần gương dài 2
L=5: nửa đầu dài 3, phần gương dài 2
L=6: nửa đầu dài 3, phần gương dài 3
99 99 18


### 3.3. `min` với `key` là tuple — xử lý "hòa thì lấy nhỏ hơn"

`min(cands, key=lambda c: (abs(c - x), c))`: so sánh theo khoảng cách trước; nếu bằng nhau thì so sánh chính `c` → số nhỏ hơn thắng. Đây là cách gọn nhất để cài quy tắc tie-break.

In [4]:
x = 1
cands = [2, 0, 11]
print(min(cands, key=lambda c: (abs(c - x), c)))
# so sánh tuple: theo phần tử đầu, bằng thì theo phần tử sau
print((1, 0) < (1, 2), (1, 5) < (2, 0))

0
True True


### 3.4. Lũy thừa của 10 và các số "biên"

- `10**(L-1) - 1` = `99...9` (`L-1` chữ số 9) — palindrome **lớn nhất có `L-1` chữ số**.
- `10**L + 1` = `100...01` (`L+1` chữ số) — palindrome **nhỏ nhất có `L+1` chữ số**.

In [5]:
for L in range(1, 6):
    print(f"L={L}: {10**(L-1) - 1:>6} | {10**L + 1:>7}")

L=1:      0 |      11
L=2:      9 |     101
L=3:     99 |    1001
L=4:    999 |   10001
L=5:   9999 |  100001


### 3.5. Tập hợp `set` — gom ứng viên không trùng, bỏ một phần tử

`set.discard(v)` xóa `v` nếu có, **không lỗi** nếu không có (khác `remove`).

In [6]:
c = {9, 11, 101, 11}
print(c)
c.discard(11); c.discard(12345)
print(c)

{9, 11, 101}
{9, 101}


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Palindrome được quyết định bởi nửa đầu

Một palindrome `L` chữ số được xác định hoàn toàn bởi `(L+1)//2` chữ số đầu. Ví dụ `L = 5`, nửa đầu `"123"` → `12321`. Và **thứ tự** các palindrome cùng độ dài **trùng với thứ tự nửa đầu**: nửa đầu lớn hơn → palindrome lớn hơn (vì nửa đầu là các chữ số có trọng số cao nhất).

### 4.2. Tính tay `n = "12932"`

- `L = 5`, nửa đầu `"129"`. Soi gương: `12921`. Cách `12932 - 12921 = 11`.
- Tăng nửa đầu: `"130"` → `13031`. Cách `99`.
- Giảm nửa đầu: `"128"` → `12821`. Cách `111`.
- Đáp án: `12921`.

### 4.3. Tính tay `n = "99"` và `n = "100"` — khi độ dài thay đổi

- `"99"`: soi gương ra `99` — chính nó, bị loại. Tăng nửa đầu `"9" → "10"` → `1001` (4 chữ số, quá xa). Giảm `"9" → "8"` → `88` (cách 11). Nhưng `101` chỉ cách **2**! `101` có **3** chữ số — không sinh ra từ nửa đầu của số 2 chữ số. → Phải thêm ứng viên `10^L + 1`.
- `"100"`: nửa đầu `"10"`. Soi gương `101` (cách 1). Giảm `"9"` → `9` (vì độ dài lẻ: `"9" + ""`) — cách 91. Nhưng `99` cách **1** và nhỏ hơn `101`! → Phải thêm ứng viên `10^(L-1) - 1`.

In [7]:
def mirror(prefix: str, L: int) -> int:
    """Dựng palindrome độ dài L từ nửa đầu prefix (prefix dài (L+1)//2)."""
    return int(prefix + prefix[:L // 2][::-1])

print(mirror("129", 5), mirror("130", 5), mirror("128", 5))
print(mirror("12", 4), mirror("9", 1), mirror("10", 3))

12921 13031 12821
1221 9 101


### 4.4. Bảng tính tay cho nhiều trường hợp

Hãy tự tính trước rồi chạy cell để so. Cột "gương" là `mirror(prefix)`; cột cuối là đáp án thật.

| n | prefix | gương | gần nhất nhỏ hơn | gần nhất lớn hơn | đáp án |
|---|---|---|---|---|---|
| 19 | 1 | 11 | 11 (cách 8) | 22 (cách 3) | 22 |
| 45 | 4 | 44 | 44 (cách 1) | 55 (cách 10) | 44 |
| 101 | 10 | 101 (chính nó) | 99 (cách 2) | 111 (cách 10) | 99 |
| 1234 | 12 | 1221 | 1221 (cách 13) | 1331 (cách 97) | 1221 |
| 9999 | 99 | 9999 (chính nó) | 9889 (cách 110) | 10001 (cách 2) | 10001 |
| 12345 | 123 | 12321 | 12321 (cách 24) | 12421 (cách 76) | 12321 |

In [8]:
def near_small(n):            # tìm lan ra hai phía (bản rút gọn của Cách 1 ở mục 5)
    x, d = int(n), 1
    while True:
        if x - d >= 0 and is_pal(x - d):
            return str(x - d)
        if is_pal(x + d):
            return str(x + d)
        d += 1

table = [("19", "22"), ("45", "44"), ("101", "99"), ("1234", "1221"), ("9999", "10001"), ("12345", "12321")]
for n, exp in table:
    L = len(n)
    pref = n[:(L + 1) // 2]
    got = near_small(n)
    print(f"n={n:>6}: prefix={pref:>4}, gương={mirror(pref, L):>6} -> đáp án {got}")
    assert got == exp

n=    19: prefix=   1, gương=    11 -> đáp án 22
n=    45: prefix=   4, gương=    44 -> đáp án 44
n=   101: prefix=  10, gương=   101 -> đáp án 99
n=  1234: prefix=  12, gương=  1221 -> đáp án 1221
n=  9999: prefix=  99, gương=  9999 -> đáp án 10001
n= 12345: prefix= 123, gương= 12321 -> đáp án 12321


## 5. Cách 1 — Brute Force: lan ra hai phía

**Ý tưởng.** Thử `d = 1, 2, 3, ...`: kiểm tra `x - d` trước (vì hòa thì lấy số nhỏ hơn), rồi `x + d`. Gặp palindrome đầu tiên là đáp án. Lưu ý `x - d` phải `>= 0`.

In [9]:
def nearest_brute(n: str) -> str:
    x = int(n)
    d = 1
    while True:
        if x - d >= 0 and is_pal(x - d):
            return str(x - d)
        if is_pal(x + d):
            return str(x + d)
        d += 1

for n in ["123", "1", "10", "11", "99", "100", "12932"]:
    print(n, "->", nearest_brute(n))
assert nearest_brute("123") == "121"
assert nearest_brute("1") == "0"

123 -> 121
1 -> 0
10 -> 9
11 -> 9
99 -> 101
100 -> 99
12932 -> 12921


**Độ phức tạp:** số bước bằng khoảng cách tới palindrome gần nhất, có thể cỡ `10^(L/2)`; mỗi bước kiểm tra `O(L)`. Với `L = 18` là `~10^9` bước → quá chậm.

**Nút thắt:** ta đi qua từng số trong khi palindrome **rất thưa**. Cần "nhảy" thẳng tới các palindrome gần nhất.

In [10]:
import time
for n in ["1000000", "123456789", "1000000000000"]:
    t = time.perf_counter()
    r = nearest_brute(n)
    print(f"{n:>14} -> {r:>14}  ({time.perf_counter() - t:.3f}s)")

       1000000 ->         999999  (0.000s)
     123456789 ->      123454321  (0.001s)
 1000000000000 ->   999999999999  (0.000s)


## 6. Bước đột phá

**Insight 1 — Palindrome cùng độ dài gần `x` nhất có nửa đầu gần nửa đầu của `x`.** Vì thứ tự palindrome cùng độ dài trùng thứ tự nửa đầu, palindrome cùng độ dài nhỏ hơn gần nhất và lớn hơn gần nhất phải có nửa đầu ∈ `{prefix - 1, prefix, prefix + 1}` (với `prefix` là nửa đầu của `x`).

Cụ thể: gọi `P = mirror(prefix)`.
- Nếu `P < x`: palindrome lớn hơn gần nhất là `mirror(prefix + 1)`, nhỏ hơn gần nhất là `P`.
- Nếu `P > x`: lớn hơn gần nhất là `P`, nhỏ hơn gần nhất là `mirror(prefix - 1)`.
- Nếu `P = x`: hai bên là `mirror(prefix ± 1)`.

**Insight 2 — Khi đổi độ dài.** Nếu `prefix ± 1` làm số chữ số của nửa đầu thay đổi (`99 → 100`, `10 → 9`), palindrome gần nhất có thể có độ dài `L - 1` hoặc `L + 1`. Mà palindrome **lớn nhất** có `L - 1` chữ số là `99...9` và **nhỏ nhất** có `L + 1` chữ số là `10...01`. Chỉ cần thêm hai số này.

**Kết luận: 5 ứng viên luôn chứa đáp án**

```text
mirror(prefix - 1), mirror(prefix), mirror(prefix + 1), 10^(L-1) - 1, 10^L + 1
```

Bỏ chính `x`, chọn ứng viên có `(|c - x|, c)` nhỏ nhất.

**Quy tắc nếu…thì…:**
- Nếu `n` là palindrome thì `mirror(prefix)` bị loại; đáp án nằm trong 4 ứng viên còn lại.
- Nếu `n = 10...0` hoặc `10...01` thì đáp án thường là `99...9` (cùng khoảng cách hoặc gần hơn, và nhỏ hơn).
- Nếu `n = 99...9` thì đáp án là `10...01` (cách 2).

In [11]:
# Kiểm chứng Insight 1: với mọi x có 4 chữ số, palindrome cùng độ dài gần nhất (nhỏ hơn / lớn hơn)
# luôn có nửa đầu thuộc {prefix-1, prefix, prefix+1}
ok = True
pals4 = [p for p in range(1000, 10000) if is_pal(p)]
for x in range(1000, 10000):
    prefix = int(str(x)[:2])
    lower = [p for p in pals4 if p < x]
    upper = [p for p in pals4 if p > x]
    for group in (lower[-1:], upper[:1]):
        for p in group:
            if int(str(p)[:2]) not in (prefix - 1, prefix, prefix + 1):
                ok = False
print("Insight 1 đúng với mọi số 4 chữ số:", ok)

Insight 1 đúng với mọi số 4 chữ số: True


## 7. Các cách tiếp theo

Notebook trình bày **3 cách** đúng (cộng thêm một cách "nửa đúng" để rút kinh nghiệm):

| # | Cách | Ý chính |
|---|---|---|
| 1 | Brute force (mục 5) | lan ra hai phía từng đơn vị |
| — | Chỉ soi gương (nửa đúng) | `mirror(prefix)` thôi — sai nhiều trường hợp |
| 2 | 5 ứng viên | `prefix - 1, prefix, prefix + 1`, `99..9`, `10..01` |
| 3 | Liền trước / liền sau | tính chính xác palindrome nhỏ hơn gần nhất và lớn hơn gần nhất |

### 7.1. Bước trung gian — chỉ soi gương (để thấy vì sao chưa đủ)

In [12]:
def nearest_mirror_only(n: str) -> str:
    L = len(n)
    p = mirror(n[:(L + 1) // 2], L)
    return str(p)

wrong = []
for x in range(1, 2000):
    if nearest_mirror_only(str(x)) != nearest_brute(str(x)):
        wrong.append(x)
print("số trường hợp sai trong 1..1999:", len(wrong))
print("vài ví dụ:", [(x, nearest_mirror_only(str(x)), nearest_brute(str(x))) for x in wrong[:6]])

số trường hợp sai trong 1..1999: 548
vài ví dụ: [(1, '1', '0'), (2, '2', '1'), (3, '3', '2'), (4, '4', '3'), (5, '5', '4'), (6, '6', '5')]


Sai ở ba kiểu: (1) `n` đã là palindrome (trả về chính nó); (2) gương nằm "xa phía bên kia" (`19 → 11` nhưng `22` gần hơn); (3) đổi độ dài (`99`, `100`).

### 7.2. Cách 2 — 5 ứng viên

**Giải thích.** Cài đúng kết luận ở mục 6. Một chi tiết: khi `prefix - 1` hoặc `prefix + 1` **đổi độ dài**, `mirror` vẫn chạy được (chỉ ra một số "lạ" như `9` hay `1001`) — không sao, vì nó chỉ là ứng viên thêm; hai số biên `10^(L-1) - 1`, `10^L + 1` đã bao các trường hợp đó.

Code demo in danh sách ứng viên:

In [13]:
def candidates(n: str):
    L = len(n)
    prefix = int(n[:(L + 1) // 2])
    cands = {10 ** (L - 1) - 1, 10 ** L + 1}
    for p in (prefix - 1, prefix, prefix + 1):
        s = str(p)
        cands.add(int(s + s[:L // 2][::-1]) if L % 2 == 0 else int(s + s[:-1][::-1]))
    return cands

def nearest_candidates(n: str) -> str:
    x = int(n)
    cands = candidates(n)
    cands.discard(x)
    return str(min(cands, key=lambda c: (abs(c - x), c)))

for n in ["123", "1", "10", "11", "99", "100", "12932", "807045053224792883"]:
    print(f"{n:>20}: ứng viên {sorted(candidates(n))} -> {nearest_candidates(n)}")
assert nearest_candidates("123") == "121"
assert nearest_candidates("1") == "0"

                 123: ứng viên [99, 111, 121, 131, 1001] -> 121
                   1: ứng viên [0, 1, 2, 11] -> 0
                  10: ứng viên [0, 9, 11, 22, 101] -> 9
                  11: ứng viên [0, 9, 11, 22, 101] -> 9
                  99: ứng viên [9, 88, 99, 101] -> 101
                 100: ứng viên [9, 99, 101, 111, 1001] -> 99
               12932: ứng viên [9999, 12821, 12921, 13031, 100001] -> 12921
  807045053224792883: ứng viên [99999999999999999, 807045052250540708, 807045053350540708, 807045054450540708, 1000000000000000001] -> 807045053350540708


**Đọc code:**

- `prefix = int(n[:(L+1)//2])` — nửa đầu (kèm chữ số giữa khi `L` lẻ).
- Dòng tạo gương: `L` chẵn → `s + s[::-1]` (vì `s[:L//2]` chính là `s` khi độ dài đúng); `L` lẻ → `s + s[:-1][::-1]` (không lặp chữ số giữa).
- Khi `prefix - 1 = 0` (ví dụ `n = "1"`), `s = "0"` → ứng viên `0`. Hợp lệ (0 là palindrome).
- `cands.discard(x)` loại chính `n`.

Lưu ý tinh tế: với `L` chẵn mà `prefix + 1` dài hơn (ví dụ `99 → 10`), `s[:L//2][::-1]` lấy `"1"` → `"10" + "1"` = `101` — vô tình cũng đúng; còn nếu viết `s + s[::-1]` sẽ ra `1001`. Cả hai đều không sai vì `10^L + 1` đã có trong tập.

**Code nộp LeetCode (Cách 2):**

In [14]:
class SolutionCandidates:
    def nearestPalindromic(self, n: str) -> str:
        L, x = len(n), int(n)
        prefix = int(n[:(L + 1) // 2])
        cands = {10 ** (L - 1) - 1, 10 ** L + 1}
        for p in (prefix - 1, prefix, prefix + 1):
            s = str(p)
            cands.add(int(s + (s[::-1] if L % 2 == 0 else s[:-1][::-1])))
        cands.discard(x)
        return str(min(cands, key=lambda c: (abs(c - x), c)))

sc = SolutionCandidates()
assert sc.nearestPalindromic("123") == "121"
assert sc.nearestPalindromic("1") == "0"
assert sc.nearestPalindromic("99") == "101"
assert sc.nearestPalindromic("100") == "99"
print("SolutionCandidates OK")

SolutionCandidates OK


**Độ phức tạp Cách 2:** 5 ứng viên, mỗi cái dựng trong `O(L)` → Time `O(L)`, Space `O(L)`. Với `L <= 18` coi như hằng số.

### 7.3. Cách 3 — Palindrome liền trước và liền sau

**Giải thích.** Thay vì "ném" 5 ứng viên rồi chọn, ta tính **chính xác** hai số:

- `nxt(x)` = palindrome **nhỏ nhất `> x`**.
- `prv(x)` = palindrome **lớn nhất `< x`**.

Đáp án = `prv(x)` nếu `x - prv(x) <= nxt(x) - x`, ngược lại `nxt(x)`. Cách này rõ ràng hơn về mặt chứng minh và dùng lại được cho bài "palindrome kế tiếp".

**`nxt(x)`:**
1. `P = mirror(prefix)`. Nếu `P > x` → trả `P`.
2. Ngược lại tăng `prefix`. Nếu `prefix + 1` dài hơn (tức `prefix = 99..9`, `x` là `99..9`) → trả `10^L + 1`. Ngược lại trả `mirror(prefix + 1)`.

**`prv(x)`:**
1. Nếu `x <= 10` → trả `x - 1` (mọi số 0..9 là palindrome).
2. `P = mirror(prefix)`. Nếu `P < x` → trả `P`.
3. Ngược lại giảm `prefix`. Nếu `prefix - 1` ngắn hơn (tức `prefix = 10..0`) **hoặc bằng 0** (trường hợp `x = 11`: nửa đầu `"1"` giảm thành `"0"`, gương `"00"` không phải số 2 chữ số) → trả `10^(L-1) - 1`. Ngược lại trả `mirror(prefix - 1)`.

In [15]:
def make_pal(half: str, L: int) -> int:
    return int(half + half[:L // 2][::-1])

def nxt(x: int) -> int:
    s = str(x); L = len(s)
    half = s[:(L + 1) // 2]
    p = make_pal(half, L)
    if p > x:
        return p
    h = str(int(half) + 1)
    if len(h) > len(half):          # half = 99..9  -> x = 99..9
        return 10 ** L + 1
    return make_pal(h, L)

def prv(x: int) -> int:
    if x <= 10:
        return x - 1
    s = str(x); L = len(s)
    half = s[:(L + 1) // 2]
    p = make_pal(half, L)
    if p < x:
        return p
    h = str(int(half) - 1)
    if len(h) < len(half) or h == "0":   # half = 10..0 (hoặc "1" khi L = 2) -> hết palindrome L chữ số nhỏ hơn
        return 10 ** (L - 1) - 1
    return make_pal(h, L)

def nearest_prev_next(n: str) -> str:
    x = int(n)
    a, b = prv(x), nxt(x)
    return str(a if x - a <= b - x else b)

for x in [1, 9, 10, 11, 12, 99, 100, 101, 110, 123, 999, 1000, 1001]:
    print(f"x={x:>5}: prv={prv(x):>5}, nxt={nxt(x):>5} -> {nearest_prev_next(str(x))}")

x=    1: prv=    0, nxt=    2 -> 0
x=    9: prv=    8, nxt=   11 -> 8
x=   10: prv=    9, nxt=   11 -> 9
x=   11: prv=    9, nxt=   22 -> 9
x=   12: prv=   11, nxt=   22 -> 11
x=   99: prv=   88, nxt=  101 -> 101
x=  100: prv=   99, nxt=  101 -> 99
x=  101: prv=   99, nxt=  111 -> 99
x=  110: prv=  101, nxt=  111 -> 111
x=  123: prv=  121, nxt=  131 -> 121
x=  999: prv=  989, nxt= 1001 -> 1001
x= 1000: prv=  999, nxt= 1001 -> 999
x= 1001: prv=  999, nxt= 1111 -> 999


**Đọc code — các nhánh biên:**

- `prv(1) = 0`, `prv(10) = 9`: xử lý trực tiếp bằng `x <= 10`.
- `prv(101)`: `half = "10"`, `P = 101` không `< 101`; `h = "9"` ngắn hơn → `99`. Đúng.
- `prv(100)`: `P = 101 > 100`; `h = "9"` ngắn hơn → `99`. Đúng.
- `prv(11)`: `half = "1"`, `P = 11` không `< 11`; `h = "0"` **cùng độ dài** nhưng gương `"00"` = 0 không phải số 2 chữ số → phải bắt riêng `h == "0"` để trả `9`. (Chính vét cạn bên dưới đã giúp phát hiện nhánh này!)
- `nxt(99)`: `P = 99` không `> 99`; `h = "10"` dài hơn → `101`. Đúng.
- `nxt(9)`: `L = 1`, `half = "9"`, `P = 9`; `h = "10"` dài hơn → `10^1 + 1 = 11`. Đúng.

Kiểm chứng `prv`, `nxt` bằng vét cạn:

In [16]:
def prv_brute(x):
    y = x - 1
    while not is_pal(y):
        y -= 1
    return y

def nxt_brute(x):
    y = x + 1
    while not is_pal(y):
        y += 1
    return y

for x in range(1, 20000):
    assert prv(x) == prv_brute(x), x
    assert nxt(x) == nxt_brute(x), x
print("prv/nxt đúng với mọi x trong 1..19999")

prv/nxt đúng với mọi x trong 1..19999


**Code nộp LeetCode (Cách 3):**

In [17]:
class SolutionPrevNext:
    def nearestPalindromic(self, n: str) -> str:
        def make(half: str, L: int) -> int:
            return int(half + half[:L // 2][::-1])

        def nxt(x: int) -> int:
            s = str(x); L = len(s); half = s[:(L + 1) // 2]
            p = make(half, L)
            if p > x:
                return p
            h = str(int(half) + 1)
            return 10 ** L + 1 if len(h) > len(half) else make(h, L)

        def prv(x: int) -> int:
            if x <= 10:
                return x - 1
            s = str(x); L = len(s); half = s[:(L + 1) // 2]
            p = make(half, L)
            if p < x:
                return p
            h = str(int(half) - 1)
            return 10 ** (L - 1) - 1 if (len(h) < len(half) or h == "0") else make(h, L)

        x = int(n)
        a, b = prv(x), nxt(x)
        return str(a if x - a <= b - x else b)

sp = SolutionPrevNext()
for n, exp in [("123", "121"), ("1", "0"), ("99", "101"), ("100", "99"), ("10", "9"), ("11", "9")]:
    assert sp.nearestPalindromic(n) == exp, n
print("SolutionPrevNext OK")

SolutionPrevNext OK


**Độ phức tạp Cách 3:** Time `O(L)`, Space `O(L)`.

## 8. TRACE từng bước

### 8.1. Trace Cách 2 trên một số ví dụ khó

| n | L | prefix | gương (p-1, p, p+1) | biên | sau khi bỏ n | đáp án |
|---|---|---|---|---|---|---|
| `123` | 3 | 12 | 111, 121, 131 | 99, 1001 | cả 5 | 121 (cách 2) |
| `99` | 2 | 9 | 88, 99, 1001 | 9, 101 | bỏ 99 | 101 (cách 2) |
| `100` | 3 | 10 | 9, 101, 111 | 99, 1001 | cả 5 | 99 (cách 1, hòa với 101, lấy nhỏ) |
| `1` | 1 | 1 | 0, 1, 2 | 0, 11 | bỏ 1 | 0 (hòa với 2) |

In [18]:
def trace_candidates(n: str) -> str:
    L, x = len(n), int(n)
    prefix = int(n[:(L + 1) // 2])
    print(f"n = {n}: L = {L}, prefix = {prefix}")
    cands = []
    for p in (prefix - 1, prefix, prefix + 1):
        s = str(p)
        v = int(s + (s[::-1] if L % 2 == 0 else s[:-1][::-1]))
        cands.append((f"mirror({p})", v))
    cands.append(("10^(L-1)-1", 10 ** (L - 1) - 1))
    cands.append(("10^L+1", 10 ** L + 1))
    best = None
    for name, v in cands:
        note = "  <- chính n, loại" if v == x else ""
        print(f"   {name:<12} = {v:<8} |c - x| = {abs(v - x)}{note}")
        if v != x and (best is None or (abs(v - x), v) < (abs(best - x), best)):
            best = v
    print(f"   => đáp án {best}\n")
    return str(best)

for n in ["123", "99", "100", "1", "12932"]:
    assert trace_candidates(n) == nearest_brute(n)

n = 123: L = 3, prefix = 12
   mirror(11)   = 111      |c - x| = 12
   mirror(12)   = 121      |c - x| = 2
   mirror(13)   = 131      |c - x| = 8
   10^(L-1)-1   = 99       |c - x| = 24
   10^L+1       = 1001     |c - x| = 878
   => đáp án 121

n = 99: L = 2, prefix = 9
   mirror(8)    = 88       |c - x| = 11
   mirror(9)    = 99       |c - x| = 0  <- chính n, loại
   mirror(10)   = 1001     |c - x| = 902
   10^(L-1)-1   = 9        |c - x| = 90
   10^L+1       = 101      |c - x| = 2
   => đáp án 101

n = 100: L = 3, prefix = 10
   mirror(9)    = 9        |c - x| = 91
   mirror(10)   = 101      |c - x| = 1
   mirror(11)   = 111      |c - x| = 11
   10^(L-1)-1   = 99       |c - x| = 1
   10^L+1       = 1001     |c - x| = 901
   => đáp án 99

n = 1: L = 1, prefix = 1
   mirror(0)    = 0        |c - x| = 1
   mirror(1)    = 1        |c - x| = 0  <- chính n, loại
   mirror(2)    = 2        |c - x| = 1
   10^(L-1)-1   = 0        |c - x| = 1
   10^L+1       = 11       |c - x| = 10
   => đáp á

### 8.2. Trace Cách 3 — in từng nhánh của `prv` và `nxt`

In [19]:
def trace_prev_next(n: str) -> str:
    x = int(n); s = str(x); L = len(s); half = s[:(L + 1) // 2]
    p = make_pal(half, L)
    print(f"x = {x}, L = {L}, half = {half}, mirror(half) = {p}")
    # nxt
    if p > x:
        b = p; why = "mirror > x"
    else:
        h = str(int(half) + 1)
        if len(h) > len(half):
            b = 10 ** L + 1; why = f"half+1 = {h} dài hơn -> 10^L+1"
        else:
            b = make_pal(h, L); why = f"mirror(half+1 = {h})"
    print(f"   nxt = {b:<6} ({why})")
    # prv
    if x <= 10:
        a = x - 1; why = "x <= 10"
    elif p < x:
        a = p; why = "mirror < x"
    else:
        h = str(int(half) - 1)
        if len(h) < len(half) or h == "0":
            a = 10 ** (L - 1) - 1; why = f"half-1 = {h} ngắn hơn (hoặc bằng 0) -> 10^(L-1)-1"
        else:
            a = make_pal(h, L); why = f"mirror(half-1 = {h})"
    print(f"   prv = {a:<6} ({why})")
    ans = a if x - a <= b - x else b
    print(f"   x - prv = {x - a}, nxt - x = {b - x} => {ans}\n")
    return str(ans)

for n in ["123", "99", "101", "1000", "12921"]:
    assert trace_prev_next(n) == nearest_brute(n)

x = 123, L = 3, half = 12, mirror(half) = 121
   nxt = 131    (mirror(half+1 = 13))
   prv = 121    (mirror < x)
   x - prv = 2, nxt - x = 8 => 121

x = 99, L = 2, half = 9, mirror(half) = 99
   nxt = 101    (half+1 = 10 dài hơn -> 10^L+1)
   prv = 88     (mirror(half-1 = 8))
   x - prv = 11, nxt - x = 2 => 101

x = 101, L = 3, half = 10, mirror(half) = 101
   nxt = 111    (mirror(half+1 = 11))
   prv = 99     (half-1 = 9 ngắn hơn (hoặc bằng 0) -> 10^(L-1)-1)
   x - prv = 2, nxt - x = 10 => 99

x = 1000, L = 4, half = 10, mirror(half) = 1001
   nxt = 1001   (mirror > x)
   prv = 999    (half-1 = 9 ngắn hơn (hoặc bằng 0) -> 10^(L-1)-1)
   x - prv = 1, nxt - x = 1 => 999

x = 12921, L = 5, half = 129, mirror(half) = 12921
   nxt = 13031  (mirror(half+1 = 130))
   prv = 12821  (mirror(half-1 = 128))
   x - prv = 100, nxt - x = 110 => 12821



### 8.3. Trace brute force — thấy "khoảng trống" giữa các palindrome

Brute force in các số đã thử. Với `n = 1000` chỉ cần 1 bước, nhưng với `n = 1050` phải lan ra 49 bước (gần 100 số được thử) mới gặp `1001` — và với số 18 chữ số khoảng trống có thể tới hàng trăm triệu. Đó là lý do phải "nhảy" bằng ứng viên.

In [20]:
def trace_brute(n, show=6):
    x, d, tried = int(n), 1, []
    while True:
        for y in (x - d, x + d):
            if y >= 0:
                tried.append(y)
                if is_pal(y):
                    head = tried[:show]
                    print(f"n={n}: thử {len(tried)} số, vài số đầu {head}{' ...' if len(tried) > show else ''} -> {y}")
                    return str(y)
        d += 1

for n in ["1000", "1050", "123456"]:
    assert trace_brute(n) == nearest_candidates(n)

n=1000: thử 1 số, vài số đầu [999] -> 999
n=1050: thử 97 số, vài số đầu [1049, 1051, 1048, 1052, 1047, 1053] ... -> 1001
n=123456: thử 269 số, vài số đầu [123455, 123457, 123454, 123458, 123453, 123459] ... -> 123321


## 9. Code nộp LeetCode

Chọn Cách 2 (5 ứng viên): ngắn, dễ nhớ, ít nhánh `if`.

In [21]:
class Solution:
    def nearestPalindromic(self, n: str) -> str:
        L, x = len(n), int(n)
        prefix = int(n[:(L + 1) // 2])
        # Hai ứng viên "đổi độ dài": 99...9 (L-1 chữ số) và 10...01 (L+1 chữ số)
        cands = {10 ** (L - 1) - 1, 10 ** L + 1}
        # Ba ứng viên soi gương từ nửa đầu - 1, nửa đầu, nửa đầu + 1
        for p in (prefix - 1, prefix, prefix + 1):
            s = str(p)
            cands.add(int(s + (s[::-1] if L % 2 == 0 else s[:-1][::-1])))
        cands.discard(x)                                   # không tính chính n
        return str(min(cands, key=lambda c: (abs(c - x), c)))

sol = Solution()
assert sol.nearestPalindromic("123") == "121"
assert sol.nearestPalindromic("1") == "0"
print(sol.nearestPalindromic("807045053224792883"))
print("Solution OK")

807045053350540708
Solution OK


## 10. Vì sao thuật toán đúng

Gọi `x` là số đầu vào, có `L` chữ số, nửa đầu `h`. Đáp án là palindrome gần nhất trong hai số: `A` = palindrome lớn nhất `< x`, `B` = palindrome nhỏ nhất `> x` (mọi palindrome khác đều xa hơn một trong hai số này). Ta chứng minh `A` và `B` đều thuộc tập 5 ứng viên.

**Chứng minh `B` thuộc tập.**
- Nếu tồn tại palindrome `L` chữ số `> x`: gọi nó là `Q` với nửa đầu `q`. Vì thứ tự palindrome `L` chữ số trùng thứ tự nửa đầu, và `mirror(h)` là palindrome duy nhất có nửa đầu `h`: nếu `mirror(h) > x` thì `B = mirror(h)`; ngược lại mọi palindrome có nửa đầu `<= h` đều `<= x`, nên `B` có nửa đầu nhỏ nhất `> h`, tức `h + 1` (nửa đầu `h + 1` vẫn là `(L+1)//2` chữ số vì `Q` tồn tại). → `B = mirror(h+1)`.
- Nếu không tồn tại (tức `x = 99...9`): `B` là palindrome nhỏ nhất có `L + 1` chữ số = `10^L + 1`.

**Chứng minh `A` thuộc tập.** Đối xứng: hoặc `A = mirror(h)`, hoặc `A = mirror(h - 1)` khi `h - 1` còn đủ số chữ số, hoặc (khi `h = 10...0`) không còn palindrome `L` chữ số nào `< x`, nên `A` = palindrome lớn nhất có `L - 1` chữ số = `10^(L-1) - 1`. Trường hợp `L = 1`: `A = x - 1 = mirror(h - 1)`.

**Tie-break.** `min` theo `(|c - x|, c)` chọn số nhỏ hơn khi hòa, đúng yêu cầu. Các ứng viên "thừa" (không phải `A`, `B`) không bao giờ gần hơn `A` hoặc `B` nên không làm sai kết quả.

## 11. Phân tích Time/Space

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Lan ra hai phía | `O(D · L)`, `D` = khoảng cách tới đáp án (tới `~10^(L/2)`) | `O(L)` | chỉ dùng kiểm chứng |
| 2. 5 ứng viên | `O(L)` | `O(L)` | **nộp bài** |
| 3. Liền trước / liền sau | `O(L)` | `O(L)` | rõ ràng, tái sử dụng được |

Vì `L <= 18`, Cách 2 và 3 thực chất là `O(1)`.

In [22]:
import random
random.seed(1)
big = [str(random.randint(1, 10**18 - 1)) for _ in range(20000)]
for name, f in [("candidates", Solution().nearestPalindromic), ("prev/next", nearest_prev_next)]:
    t = time.perf_counter()
    for n in big:
        f(n)
    print(f"{name:>10}: 20000 số 18 chữ số trong {time.perf_counter() - t:.3f}s")

candidates: 20000 số 18 chữ số trong 0.051s
 prev/next: 20000 số 18 chữ số trong 0.022s


## 12. Bộ test

In [23]:
impls = {
    "candidates": nearest_candidates,
    "prev_next": nearest_prev_next,
    "SolutionCandidates": SolutionCandidates().nearestPalindromic,
    "SolutionPrevNext": SolutionPrevNext().nearestPalindromic,
    "Solution": Solution().nearestPalindromic,
}

# Ví dụ đề
for name, f in impls.items():
    assert f("123") == "121" and f("1") == "0", name

# Edge case: đổi độ dài, palindrome sẵn, số 10..0, 99..9, 10..01, 11
edge = {
    "9": "8", "10": "9", "11": "9", "99": "101", "100": "99", "101": "99",
    "999": "1001", "1000": "999", "1001": "999", "88": "77", "121": "111",
    "1213": "1221", "10000": "9999", "99999": "100001",
    "999999999999999999": "1000000000000000001",
    "100000000000000000": "99999999999999999",
}
for n, exp in edge.items():
    assert nearest_brute(n) == exp if len(n) < 8 else True, n
    for name, f in impls.items():
        assert f(n) == exp, (name, n, f(n))
print("Edge cases OK")

# Duyệt toàn bộ 1..30000, đối chiếu brute force
for x in range(1, 30001):
    b = nearest_brute(str(x))
    for name, f in impls.items():
        assert f(str(x)) == b, (name, x)
print("Khớp brute force với mọi n trong 1..30000")

# Ngẫu nhiên có seed, số tới 10^8
random.seed(564)
for _ in range(300):
    x = random.randint(1, 10**8)
    b = nearest_brute(str(x))
    assert all(f(str(x)) == b for f in impls.values()), x
print("300 test ngẫu nhiên tới 10^8 khớp")

# Số lớn: các cách nhanh khớp nhau
for n in big[:5000]:
    assert len({f(n) for f in impls.values()}) == 1, n
print("5000 số 18 chữ số: các cách khớp nhau")

Edge cases OK


Khớp brute force với mọi n trong 1..30000
300 test ngẫu nhiên tới 10^8 khớp


5000 số 18 chữ số: các cách khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên loại chính `n` khi `n` đã là palindrome

In [24]:
def wrong_keep_self(n):
    L, x = len(n), int(n)
    prefix = int(n[:(L + 1) // 2])
    cands = {10 ** (L - 1) - 1, 10 ** L + 1}
    for p in (prefix - 1, prefix, prefix + 1):
        s = str(p)
        cands.add(int(s + (s[::-1] if L % 2 == 0 else s[:-1][::-1])))
    return str(min(cands, key=lambda c: (abs(c - x), c)))   # QUÊN discard(x)

print("sai:", wrong_keep_self("121"), "| đúng:", Solution().nearestPalindromic("121"))

sai: 121 | đúng: 111


**Vì sao sai:** đề nói "không tính chính nó". Không loại thì khoảng cách 0 luôn thắng.

**Code đúng:** `cands.discard(x)` trước khi chọn.

### Lỗi 2 — Thiếu hai ứng viên đổi độ dài

In [25]:
def wrong_no_boundary(n):
    L, x = len(n), int(n)
    prefix = int(n[:(L + 1) // 2])
    cands = set()
    for p in (prefix - 1, prefix, prefix + 1):
        s = str(p)
        cands.add(int(s + (s[::-1] if L % 2 == 0 else s[:-1][::-1])))
    cands.discard(x)
    return str(min(cands, key=lambda c: (abs(c - x), c)))

for n in ["99", "100", "1000", "11"]:
    print(n, "sai:", wrong_no_boundary(n), "| đúng:", Solution().nearestPalindromic(n))

99 sai: 88 | đúng: 101
100 sai: 101 | đúng: 99
1000 sai: 1001 | đúng: 999
11 sai: 0 | đúng: 9


**Vì sao sai:** `99 → 101` và `1000 → 999` cần palindrome **khác độ dài**. Gương của `prefix ± 1` không cho ra đúng các số đó (`99`: `prefix+1 = 10` → `1001`; `1000`: `prefix-1 = 9` → `99`).

**Code đúng:** thêm `10^(L-1) - 1` và `10^L + 1`.

### Lỗi 3 — Tie-break sai (lấy số lớn hơn, hoặc phụ thuộc thứ tự duyệt `set`)

In [26]:
def wrong_tie(n):
    L, x = len(n), int(n)
    prefix = int(n[:(L + 1) // 2])
    cands = {10 ** (L - 1) - 1, 10 ** L + 1}
    for p in (prefix - 1, prefix, prefix + 1):
        s = str(p)
        cands.add(int(s + (s[::-1] if L % 2 == 0 else s[:-1][::-1])))
    cands.discard(x)
    return str(min(cands, key=lambda c: abs(c - x)))     # chỉ so khoảng cách

bad = [x for x in range(1, 3000) if wrong_tie(str(x)) != Solution().nearestPalindromic(str(x))]
print("số trường hợp sai trong 1..2999:", len(bad), "| ví dụ:", bad[:5])

số trường hợp sai trong 1..2999: 96 | ví dụ: [22, 33, 55, 66, 88]


**Vì sao sai (hoặc đúng "nhờ may"):** `min` trả phần tử **đầu tiên** đạt min theo thứ tự duyệt của `set` — thứ tự này không được đảm bảo. Có thể đúng trên máy bạn nhưng sai trên máy khác/phiên bản khác.

**Code đúng:** `key=lambda c: (abs(c - x), c)`.

### Lỗi 4 — Dùng `s + s[::-1]` cho cả độ dài lẻ

In [27]:
def wrong_odd_mirror(n):
    L = len(n)
    s = n[:(L + 1) // 2]
    return s + s[::-1]          # lẻ: lặp chữ số giữa -> độ dài L+1

print("'12345' -> gương sai:", wrong_odd_mirror("12345"), "| đúng:", "123" + "12"[::-1])

'12345' -> gương sai: 123321 | đúng: 12321


**Vì sao sai:** với `L` lẻ, chữ số giữa chỉ xuất hiện **một lần**. `s + s[::-1]` tạo số dài `L + 1`.

**Code đúng:** `s + s[:-1][::-1]` khi `L` lẻ (hoặc gọn hơn: `s + s[:L//2][::-1]` cho cả hai trường hợp).

### Lỗi 5 — Dùng số thực `float` cho số 18 chữ số

In [28]:
n = "999999999999999999"
x_float = float(n)
print(int(x_float), "| đúng:", int(n))
print("float phân biệt được 10^17 và 10^17 + 1?", float(10**17) != float(10**17 + 1))

1000000000000000000 | đúng: 999999999999999999
float phân biệt được 10^17 và 10^17 + 1? False


**Vì sao sai:** `float` chỉ có ~15–16 chữ số có nghĩa. `float("999999999999999999")` làm tròn thành `10^18` — sai ngay từ đầu. Các phép như `math.pow(10, L)` cũng trả `float`.

**Code đúng:** chỉ dùng `int(...)` và `10 ** L` (số nguyên chính xác tùy ý trong Python).

## 14. Mẹo & nhận dạng pattern

- **"Tìm số đặc biệt gần nhất"** mà số đặc biệt rất thưa (palindrome, số có chữ số tăng dần...) → **đừng tìm kiếm, hãy liệt kê ứng viên** quanh `x` theo cấu trúc của số đặc biệt.
- **Palindrome = nửa đầu + gương.** Mọi bài palindrome số đều bắt đầu từ đây: đếm palindrome `<= x`, palindrome thứ `k`, palindrome kế tiếp...
- **Biên "đổi độ dài"**: `99..9 ↔ 10..01`. Luôn thử `9`, `10`, `11`, `99`, `100`, `101`, `999`, `1000`, `1001`.
- **Tie-break** "hòa thì lấy nhỏ hơn" → `key=(khoảng cách, giá trị)`.
- Khi bài có nhiều nhánh biên, **duyệt hết** mọi input nhỏ (ví dụ 1..30000) đối chiếu brute force — rẻ và cực kỳ hiệu quả.

## 15. Biến thể

1. **Palindrome kế tiếp (`> x`)**: chính là `nxt(x)` ở Cách 3.
2. **Palindrome gần nhất, được phép là chính `n`**: bỏ `discard`.
3. **Đếm palindrome trong `[1, x]`** (Bài tập 5).
4. **LeetCode 906 — Super Palindromes**: liệt kê palindrome qua nửa đầu, kiểm tra bình phương.
5. **LeetCode 2217 — Find Palindrome With Fixed Length**: palindrome thứ `k` có độ dài cố định = `mirror(10^(h-1) + k - 1)`.

In [29]:
# Biến thể 5: palindrome thứ k (1-indexed) có độ dài L
def kth_palindrome(k, L):
    h = (L + 1) // 2
    half = 10 ** (h - 1) + k - 1
    if len(str(half)) > h:
        return -1
    return make_pal(str(half), L)

print([kth_palindrome(k, 3) for k in range(1, 12)])
assert [kth_palindrome(k, 3) for k in range(1, 91)] == [p for p in range(100, 1000) if is_pal(p)]
assert kth_palindrome(91, 3) == -1

[101, 111, 121, 131, 141, 151, 161, 171, 181, 191, 202]


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Lan ra hai phía | thử `x-d`, `x+d` với `d = 1, 2, ...` | `O(D · L)` | `O(L)` | Kiểm chứng, `n` nhỏ |
| — Chỉ soi gương | `mirror(prefix)` | `O(L)` | `O(L)` | **Không dùng** — sai biên |
| 2. 5 ứng viên | gương `prefix-1..prefix+1` + `99..9`, `10..01` | `O(L)` | `O(L)` | **Nộp bài**, ít code |
| 3. Liền trước / liền sau | tính chính xác `prv(x)`, `nxt(x)` | `O(L)` | `O(L)` | Cần cả hai phía, dễ chứng minh |

## 17. Bài tập

**Bài 1 (đoán output).** Không chạy code: đáp án cho `"1001"`, `"10"`, `"808"`?

**Bài 2 (tính tay).** Liệt kê 5 ứng viên của Cách 2 cho `n = "4567"` và chọn đáp án.

**Bài 3 (điền TODO).** Hoàn thiện `mirror_any(half, L)` dựng palindrome độ dài `L` từ nửa đầu (đúng cho cả `L` chẵn, lẻ).

**Bài 4 (tự viết lại).** Tự viết `next_palindrome(x)` (palindrome nhỏ nhất `> x`) mà không nhìn Cách 3.

**Bài 5 (mini challenge).** Viết `count_palindromes(x)` = số palindrome trong `[1, x]`, chạy `O(L)`, đúng với `x` tới `10^18`.

In [30]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def mirror_any(half: str, L: int) -> int:
    # TODO: ghép half với phần đảo của half[:L//2]
    pass

# Bài 4 — TODO
def next_palindrome(x: int) -> int:
    # TODO: thử mirror(half); nếu không > x thì tăng half; xử lý 99..9
    pass

# Bài 5 — TODO
def count_palindromes(x: int) -> int:
    # TODO: cộng số palindrome mọi độ dài < L, rồi đếm palindrome L chữ số <= x
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `1001` là palindrome — phải loại nó.
- Mức 2: so sánh `999` và `1111`; `10`: so `9` và `11`; `808`: so `797` và `818`.
- Mức 3: tính đúng từng khoảng cách: `808 - 797 = 11`, `818 - 808 = 10`; chỉ khi hòa mới lấy số nhỏ.

**Bài 2.**
- Mức 1: `L = 4`, `prefix = 45`.
- Mức 2: gương: `4444`, `4554`, `4664`; biên: `999`, `10001`.
- Mức 3: khoảng cách tới 4567: `123`, `13`, `97`, ...

**Bài 3.**
- Mức 1: phần gương dài `L // 2`.
- Mức 2: `half[:L // 2]` rồi đảo.
- Mức 3: `int(half + half[:L//2][::-1])`.

**Bài 4.**
- Mức 1: lấy nửa đầu `half` của `x`.
- Mức 2: `p = mirror(half)`; nếu `p > x` trả `p`.
- Mức 3: `half + 1` dài hơn → `10^L + 1`.

**Bài 5.**
- Mức 1: số palindrome có đúng `d` chữ số = `9 · 10^(ceil(d/2) - 1)`.
- Mức 2: palindrome `L` chữ số `<= x`: các nửa đầu từ `10^(h-1)` tới `prefix - 1` đều được.
- Mức 3: cộng thêm 1 nếu `mirror(prefix) <= x`.

## 19. Đáp án tham khảo

**Bài 1:** `"1001" → "999"` (cách 2; `1111` cách 110), `"10" → "9"`, `"808" → "818"` (818 cách 10, còn 797 cách 11 — cẩn thận, không phải hòa!).

**Bài 2:** ứng viên `{4444, 4554, 4664, 999, 10001}`, đáp án `4554` (cách 13).

In [31]:
assert Solution().nearestPalindromic("1001") == "999"
assert Solution().nearestPalindromic("10") == "9"
assert Solution().nearestPalindromic("808") == "818"
assert candidates("4567") == {4444, 4554, 4664, 999, 10001}
assert Solution().nearestPalindromic("4567") == "4554"

# Bài 3
def mirror_any(half: str, L: int) -> int:
    return int(half + half[:L // 2][::-1])

assert mirror_any("123", 5) == 12321 and mirror_any("12", 4) == 1221 and mirror_any("7", 1) == 7

# Bài 4
def next_palindrome(x: int) -> int:
    s = str(x); L = len(s); half = s[:(L + 1) // 2]
    p = mirror_any(half, L)
    if p > x:
        return p
    h = str(int(half) + 1)
    if len(h) > len(half):
        return 10 ** L + 1
    return mirror_any(h, L)

for x in range(0, 20000):
    assert next_palindrome(x) == nxt_brute(x), x
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [32]:
# Bài 5
def count_palindromes(x: int) -> int:
    if x <= 0:
        return 0
    s = str(x); L = len(s)
    total = 0
    for d in range(1, L):                         # mọi độ dài nhỏ hơn L
        total += 9 * 10 ** ((d + 1) // 2 - 1)
    h = (L + 1) // 2
    prefix = int(s[:h])
    total += prefix - 10 ** (h - 1)               # nửa đầu nhỏ hơn prefix
    if mirror_any(s[:h], L) <= x:                 # chính nửa đầu prefix
        total += 1
    return total

running = 0
for x in range(1, 30000):
    running += is_pal(x)
    assert count_palindromes(x) == running, x
print("Bài 5 OK; số palindrome trong [1, 10^18 - 1]:", count_palindromes(10**18 - 1))
assert count_palindromes(10**18 - 1) == 2 * (10**9 - 1)

Bài 5 OK; số palindrome trong [1, 10^18 - 1]: 1999999998


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "palindrome gần nhất":**

```python
L, x = len(n), int(n)
h = int(n[:(L + 1) // 2])
cands = {10**(L-1) - 1, 10**L + 1}
for p in (h - 1, h, h + 1):
    s = str(p)
    cands.add(int(s + s[:L // 2][::-1]))   # hoặc tách chẵn/lẻ cho rõ
cands.discard(x)
return str(min(cands, key=lambda c: (abs(c - x), c)))
```

**Câu thần chú:** *"Nửa đầu ±1, cộng hai số biên 99..9 và 10..01, bỏ chính nó, hòa lấy nhỏ."*

**Checklist trước khi submit:**

- [ ] Nửa đầu dài `(L + 1) // 2`; gương dài `L // 2`.
- [ ] Đủ 5 ứng viên, có `10^(L-1) - 1` và `10^L + 1`.
- [ ] Loại chính `n` (`discard`).
- [ ] Tie-break `(khoảng cách, giá trị)`.
- [ ] Trả về **chuỗi**.
- [ ] Thử `"1" → "0"`, `"10" → "9"`, `"11" → "9"`, `"99" → "101"`, `"100" → "99"`, `"1001" → "999"`.
- [ ] Đối chiếu brute force với mọi `n` nhỏ.